## 2. Llevar datos divididos y crudos a la base de datos principal SQL SERVER

In [7]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# 1. Configurar la conexión a SQL Server con fast_executemany=True
servidor = r'DESKTOP-5LI3A1D\SQLEXPRESS'
base_datos = 'Manufactura'

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={servidor};"
    f"DATABASE={base_datos};"
    f"Trusted_Connection=yes;"
)

# Motor de conexión optimizado para lotes en SQL Server
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

# 2. Cargar TODAS las hojas del archivo de Excel normalizado
archivo_excel = 'manufactura_normalizada.xlsx'

try:
    todas_las_hojas = pd.read_excel(archivo_excel, sheet_name=None)
    print(f"Hojas encontradas en el Excel: {list(todas_las_hojas.keys())}")

    # 3. Definir el orden de carga para respetar las llaves foráneas
    orden_carga = ['Catalogo_Maquinas', 'Catalogo_Productos', 'Metricas_Produccion']

    for nombre_tabla in orden_carga:
        if nombre_tabla in todas_las_hojas:
            df_actual = todas_las_hojas[nombre_tabla]
            
            # Limpiar espacios en los nombres de las columnas
            df_actual.columns = df_actual.columns.str.strip()

            # 4. SOLUCIÓN AL ERROR DE CLAVE PRIMARIA: Eliminar duplicados en origen
            if nombre_tabla == 'Catalogo_Maquinas' and 'machine_id' in df_actual.columns:
                df_actual = df_actual.drop_duplicates(subset=['machine_id'])
            elif nombre_tabla == 'Catalogo_Productos' and 'product_type' in df_actual.columns:
                df_actual = df_actual.drop_duplicates(subset=['product_type'])

            print(f"Insertando datos en la tabla '{nombre_tabla}'...")
            
            # 5. Cargar los datos a SQL Server por bloques (chunksize)
            df_actual.to_sql(
                name=nombre_tabla, 
                con=engine, 
                schema='dbo', 
                if_exists='append',  # Inserta en la tabla existente sin borrar estructura
                index=False,
                chunksize=500
            )
            print(f"-> ¡Tabla '{nombre_tabla}' cargada con éxito ({len(df_actual)} registros)!")
        else:
            print(f"-> Advertencia: La hoja '{nombre_tabla}' no se encontró en el archivo Excel.")

    print("\n¡Proceso de carga de múltiples tablas finalizado con éxito!")

except Exception as e:
    print(f"\nOcurrió un error durante la carga a SQL Server:\n{e}")

Hojas encontradas en el Excel: ['Metricas_Produccion', 'Catalogo_Maquinas', 'Catalogo_Productos']
Insertando datos en la tabla 'Catalogo_Maquinas'...
-> ¡Tabla 'Catalogo_Maquinas' cargada con éxito (20 registros)!
Insertando datos en la tabla 'Catalogo_Productos'...
-> ¡Tabla 'Catalogo_Productos' cargada con éxito (4 registros)!
Insertando datos en la tabla 'Metricas_Produccion'...
-> ¡Tabla 'Metricas_Produccion' cargada con éxito (15000 registros)!

¡Proceso de carga de múltiples tablas finalizado con éxito!


In [3]:
try:
        manufactura_normalizada.to_sql('Metricas_Produccion', con=engine, if_exists='append', index=False)
    print("¡Datos cargados exitosamente!")
except Exception as e:
    print(f"Ocurrió un error al cargar los datos: {e}")

IndentationError: unindent does not match any outer indentation level (<string>, line 3)